In [8]:
#Points to make GB algorithm
#1. Initial prediction
#2. residual = target - prediction
#   trees = []
#3. for loop
#  4. Desicion Tree
#  5. Train (X, Residuals)
#  6. predictions = predict(X)
#  7. residuals = residuals - lr * predictions
#  8. trees.append(tree)


#Initial prediction is taken by doin mean of targets
#residual = Target-pred
#Train DT with Train, residual
#prediction = predicted residual
#new prediction = old prediction + learning rate * predicted residual
#append DT in an array
#repeat from step 2 for all the trees

In [9]:
import numpy as np

x = np.array([[1,2],
             [2,3],
             [3,4],
             [4,5],
             [5,6]])

y = np.array([1, 1, 0, 0, 0])

In [10]:
class DecisionTree:
  class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
      if self.value is not None:
        return True
      else:
        return False


  def __init__(self, max_depth = 3):
    self.max_depth = max_depth
    self.root = None
    self.best_gain = float("inf")
    self.split_index = None
    self.split_threshold = None
    self.best_feature = None
    self.best_threshold = None

  def fit(self, X, y):
    self.root = self.build_tree(X, y)

  def Mean_of_Labels(self, y):
      return np.mean(y)

  def Split(self, X_column, split_threshold):
    left_indices = np.argwhere(X_column <= split_threshold).flatten()
    right_indices = np.argwhere(X_column > split_threshold).flatten()
    return left_indices, right_indices

  def Entropy(self, y):
    y_bar = np.mean(y)
    entropy = np.mean((y - y_bar)**2)
    return entropy

  def Information_Gain(self, y, X_column, split_threshold):
    left_indices, right_indices = self.Split(X_column, split_threshold)
    if len(left_indices) == 0 or len(right_indices) == 0:
      return float("inf")
    e_left = self.Entropy(y[left_indices])
    e_right = self.Entropy(y[right_indices])
    nl = len(left_indices)
    nr = len(right_indices)
    n = len(y)
    weighted_entropy = (nl / n) * e_left + (nr / n) * e_right
    return weighted_entropy

  def Best_Split(self, X, y):
    best_gain = float("inf")
    split_index = None
    split_threshold = None

    for feature_index in range(X.shape[1]):
      X_column = X[:, feature_index]
      X_column_sorted = np.sort(X_column)
      thresholds = (X_column_sorted[:-1] + X_column_sorted[1:])/2
      for threshold in thresholds:
        gain = self.Information_Gain(y, X_column, threshold)
        if gain < best_gain:
          best_gain = gain
          split_index = feature_index
          split_threshold = threshold
    return split_index, split_threshold


  def build_tree(self, X, y, depth = 0):
    n_labels = len(np.unique (y))
    if (n_labels == 1 or depth == self.max_depth):
      leaf_value = self.Mean_of_Labels(y)
      return self.Node(value=leaf_value)

    best_feature, best_threshold = self.Best_Split(X, y)

    if best_feature is None:
        leaf_value = self.Mean_of_Labels(y)
        return self.Node(value=leaf_value)

    left_indices, right_indices = self.Split(X[:, best_feature], best_threshold)

    if len(left_indices) == 0 or len(right_indices) == 0:
        leaf_value = self.Mean_of_Labels(y)
        return self.Node(value=leaf_value)


    left_subtree = self.build_tree(X[left_indices, :], y[left_indices], depth+1)
    right_subtree = self.build_tree(X[right_indices, :], y[right_indices], depth+1)
    return self.Node(best_feature, best_threshold, left_subtree, right_subtree)


  def traverse_tree(self, x, node):
    if node.is_leaf():
      return node

    if x[node.feature] <= node.threshold:
      return self.traverse_tree(x, node.left)
    else:
      return self.traverse_tree(x, node.right)

  def predict(self, X):
    predictions = []
    for x_row in X:
      predicted_node = self.traverse_tree(x_row, self.root)
      predictions.append(predicted_node.value)
    return np.array(predictions)

In [11]:
class GradientBoosting:
  def __init__(self, n_estimators, learning_rate = 0.1):
    self.n_estimators = n_estimators
    self.trees = []
    self.learning_rate = learning_rate

  def fit(self, X, y):
    self.initial_prediction = np.mean(y)
    y_pred = np.full(y.shape, self.initial_prediction)
    for i in range(self.n_estimators):
      residuals = y - y_pred
      tree = DecisionTree()
      tree.fit(X, residuals)
      predicted_residuals = tree.predict(X)
      y_pred = y_pred + self.learning_rate * predicted_residuals
      self.trees.append(tree)

  def predict(self, X):
    y_pred = np.full(X.shape[0], self.initial_prediction)
    for i in range(self.n_estimators):
      tree = self.trees[i]
      prediction = tree.predict(X)
      y_pred = y_pred + self.learning_rate * prediction
    return y_pred

In [12]:
model = GradientBoosting(n_estimators=100, learning_rate = 0.1)
model.fit(x, y)
predictions = model.predict(x)
print(predictions)

[9.99984063e-01 9.99984063e-01 1.06245596e-05 1.06245596e-05
 1.06245596e-05]


In [13]:
def R2score(y, y_pred):
  TSS = np.sum(y-np.mean(y)**2)
  RSS = np.sum((y-y_pred)**2)
  R2 = 1-(RSS/TSS)
  return R2

print(R2score(y, predictions)*100)

99.99999992944922


In [14]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

model = GradientBoostingRegressor(n_estimators=100, learning_rate = 0.1)
model.fit(x, y)
predictions = model.predict(x)
print("Predictions:", predictions)

print("R2score:", r2_score(y, predictions)*100)


Predictions: [9.99984063e-01 9.99984063e-01 1.06245596e-05 1.06245596e-05
 1.06245596e-05]
R2score: 99.99999992944922
